# Pipeline CRM Atlas — Glue / Spark

Versión Spark del pipeline CRM. Lee todas las fuentes directamente desde Silver S3, procesa y escribe reportes en Gold S3.

In [ ]:
%idle_timeout 720
%glue_version 5.1
%worker_type G.1X
%number_of_workers 5


In [ ]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import *
from datetime import datetime, date, timedelta
import boto3

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
spark.conf.set('spark.sql.session.timeZone', 'America/Mexico_City')
print('Spark session iniciada.')


In [ ]:
BUCKET      = 'aws-s3-reporte-atenea-usorg-nprd'
BASE        = f's3://{BUCKET}/data/'
GOLD_BASE   = f's3://{BUCKET}/data/Gold/Drive/'
FECHA_CORTE = '2026-07-01'

SOURCES = {
    'historico_oportunidades': 'Silver/Atlas/ProcHistoricoOportunidades/',
    'citas':                   'Silver/Atlas/ProcCitas/',
    'oportunidades':           'Silver/Atlas/ProcOportunidades/',
    'historico_citas':         'Silver/Atlas/ProcHistoricoCitas/',
    'historico_casos':         'Silver/Atlas/ProcCasosCrmHistorico/',
    'casos':                   'Silver/Atlas/ProcCasosCrm/',
    'pedidos':                 'Silver/Atlas/ProcPedidos/',
    'solicitudes_credito':     'Silver/Atlas/ProcSolicitudesCredito/',
    'catalogo_usuarios':       'Silver/Atlas/ProcCatalogoUsuarios/',
    'clientes':                'Silver/Atlas/ProcClientes/',
}


## 1. Lectura de fuentes desde Silver DL

In [ ]:
def latest_partition(base_s3_path):
    """Regresa el path de la particion mas reciente year/month/day."""
    s3 = boto3.client('s3')
    bucket, prefix = base_s3_path.replace('s3://', '').split('/', 1)
    prefix = prefix.rstrip('/') + '/'

    def max_partition(pfx, key):
        resp = s3.list_objects_v2(Bucket=bucket, Prefix=pfx, Delimiter='/')
        parts = [
            (int(p['Prefix'].rstrip('/').split('=')[-1]), p['Prefix'])
            for p in resp.get('CommonPrefixes', [])
            if '=' in p['Prefix']
        ]
        if not parts:
            raise FileNotFoundError(f'No partitions ({key}=) under {pfx}')
        return max(parts)[1]

    year_pfx  = max_partition(prefix,   'year')
    month_pfx = max_partition(year_pfx, 'month')
    day_pfx   = max_partition(month_pfx, 'day')
    full = f's3://{bucket}/{day_pfx}'
    print(f'  -> {full}')
    return full


In [ ]:
_t0_total = datetime.now()
_dfs = {}

for nombre, path in SOURCES.items():
    _t0 = datetime.now()
    print(f'Leyendo {nombre}...')
    full_path = latest_partition(BASE + path)
    _dfs[nombre] = spark.read.parquet(full_path)
    print(f'  -> {_dfs[nombre].count():,} filas en {(datetime.now()-_t0).seconds}s')

historico_oportunidades = _dfs['historico_oportunidades']
citas                   = _dfs['citas']
oportunidades           = _dfs['oportunidades']
historico_citas         = _dfs['historico_citas']
historico_casos         = _dfs['historico_casos']
casos                   = _dfs['casos']
pedidos                 = _dfs['pedidos']
solicitudes_credito     = _dfs['solicitudes_credito']
catalogo_usuarios       = _dfs['catalogo_usuarios']
clientes                = _dfs['clientes']

print(f'Lecturas completadas en {(datetime.now()-_t0_total).seconds}s')


In [ ]:
def filtrar_fecha(df, col, corte=FECHA_CORTE):
    if col not in df.columns:
        print(f'  [SKIP] columna {col!r} no encontrada — sin filtro de fecha')
        return df
    return df.filter(F.col(col) > F.lit(corte))

historico_oportunidades = filtrar_fecha(historico_oportunidades, 'created_date')
citas                   = filtrar_fecha(citas,                   'created_date')
oportunidades           = filtrar_fecha(oportunidades,           'opportunity_created_date')
historico_citas         = filtrar_fecha(historico_citas,         'created_date')
historico_casos         = filtrar_fecha(historico_casos,         'case_created_date')
casos                   = filtrar_fecha(casos,                   'case_created_date')


In [ ]:
# Si el catalogo no trae periodos SCD2, se agregan como siempre-vigentes
if 'start_date' not in catalogo_usuarios.columns:
    catalogo_usuarios = (
        catalogo_usuarios
        .withColumn('start_date', F.lit('2000-01-01').cast('date'))
        .withColumn('end_date',   F.lit('2099-12-31').cast('date'))
    )


## 2. Proceso — Reportes consumibles

In [ ]:
# Deduplicar catalogo: 1 fila por usuario (la mas reciente por processing_date)
# Necesario porque el catalogo tiene un snapshot diario y cada snapshot genera una fila.
# Sin dedup, el join SCD2 produce fan-out (N filas por oportunidad).
_w_cat = Window.partitionBy('id').orderBy(F.col('processing_date').desc())
_cat_dedup = (
    catalogo_usuarios
    .withColumn('_rn', F.row_number().over(_w_cat))
    .filter(F.col('_rn') == 1)
    .drop('_rn')
)
_cat = (
    _cat_dedup
    .withColumnRenamed('id',   'cat_id')
    .withColumnRenamed('name', 'cat_nombre')
    .withColumn('start_date', F.lit('2000-01-01').cast('date'))
    .withColumn('end_date',   F.lit('2099-12-31').cast('date'))
    .select('cat_id', 'cat_nombre', 'equipo', 'start_date', 'end_date')
)

# Pedidos a 1 fila por opportunity (el mas reciente).
# Sin esto, una oportunidad con multiples pedidos genera N filas al joinear.
_w_ped = Window.partitionBy('opportunity_id').orderBy(F.col('fecha_de_creacion').desc())
_pedidos_1x1 = (
    pedidos
    .withColumn('_rn', F.row_number().over(_w_ped))
    .filter(F.col('_rn') == 1)
    .drop('_rn')
    .select('opportunity_id', 'sf_order_id', 'sf_order_status')
    .alias('ped')
)

# Casos a 1 fila por opportunity (el mas reciente).
_w_cas = Window.partitionBy('opportunity_id').orderBy(F.col('case_created_date').desc())
_casos_1x1 = (
    casos
    .withColumn('_rn', F.row_number().over(_w_cas))
    .filter(F.col('_rn') == 1)
    .drop('_rn')
    .select('opportunity_id', 'case_id', 'case_status')
    .alias('cas')
)

# Reporte oportunidades: join principal con pedido 1x1, caso 1x1 y equipo del asesor (SCD2)
oportunidades_consumo = (
    oportunidades
    .join(_pedidos_1x1, on='opportunity_id', how='left')
    .join(_casos_1x1,   on='opportunity_id', how='left')
    .join(
        _cat,
        on=(
            (F.col('owner_id')                 == F.col('cat_id')) &
            (F.col('opportunity_created_date') >= F.col('start_date')) &
            (F.col('opportunity_created_date') <= F.col('end_date'))
        ),
        how='left'
    )
)
oportunidades_consumo = oportunidades_consumo.withColumnRenamed('cat_nombre', 'nombre_asesor')
print(f'oportunidades_consumo: {oportunidades_consumo.count():,} filas')


In [ ]:
# Reporte historico: cambios de etapa enriquecidos con fecha de creacion de la oportunidad
reporte_historico_oportunidades = (
    historico_oportunidades
    .join(
        oportunidades.select('opportunity_id', 'opportunity_created_date'),
        on='opportunity_id', how='left'
    )
)
print(f'reporte_historico_oportunidades: {reporte_historico_oportunidades.count():,} filas')


In [ ]:
# Deduplicar oportunidades_consumo a 1 fila por opportunity_id antes de joinear con citas.
# Sin esto, si una oportunidad tiene N filas residuales, cada cita se multiplica por N.
_w_opc = Window.partitionBy('opportunity_id').orderBy(F.lit(1))
_opps_citas = (
    oportunidades_consumo
    .withColumn('_rn', F.row_number().over(_w_opc))
    .filter(F.col('_rn') == 1)
    .drop('_rn')
    .select('opportunity_id', 'nombre_asesor', 'equipo')
)

# Reporte citas:
# - oportunidad: por opportunity_id (1x1 ya garantizado)
# - pedido:      por commerce_order_id (cada cita puede tener su propio pedido)
# - cliente:     por id_am de citas == id_am de clientes (nickname del comprador)
reporte_citas = (
    citas
    .join(_opps_citas, on='opportunity_id', how='left')
    .join(
        pedidos.select('commerce_order_id', 'sf_order_id'),
        on='commerce_order_id', how='left'
    )
    .join(
        clientes.select(
            F.col('id_am').alias('cliente_id_am'),
            F.concat_ws(' ', F.col('billing_firstname'), F.col('billing_lastname')).alias('nombre_cliente')
        ),
        on=citas['id_am'] == F.col('cliente_id_am'), how='left'
    )
    .drop('cliente_id_am')
)
print(f'reporte_citas: {reporte_citas.count():,} filas')


In [ ]:
reporte_simulaciones = solicitudes_credito
print(f'reporte_simulaciones: {reporte_simulaciones.count():,} filas')


## 3. Escritura en Gold S3

In [ ]:
def write_gold(df, nombre):
    path = GOLD_BASE + nombre + '/'
    df.write.mode('overwrite').parquet(path)
    print(f'  Escrito {nombre} -> {path}')

write_gold(oportunidades_consumo,           'ReporteOportunidades')
write_gold(reporte_historico_oportunidades, 'ReporteHistoricoOportunidades')
write_gold(reporte_citas,                   'ReporteCitas')
write_gold(reporte_simulaciones,            'ReporteSimulaciones')


In [ ]:
print(f'Pipeline CRM Glue completado: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
